In [ ]:
import numpy as np
import scipy.integrate
from perfetto.trace_processor import TraceProcessor
import pandas as pd
from typing import Dict, Any, Optional

In [ ]:
class LBO(object):
    GC_SLICE_SQL_LIKE = "% % GC"

    def __init__(self, trace_path: str):
        self.tp = TraceProcessor(trace=trace_path)
        self.__create_thread_state_int_view()
        self.__create_freq_counter_view()
        self.__create_gc_slice_view()
        self.__create_gc_slice_cpu_time_view()
        self.__create_thread_cpu_time_view()
        self.__create_cycles_view("thread_cycles", "thread_cpu_time")
        self.__create_cycles_view("gc_slice_cycles", "gc_slice_cpu_time")
    
    def __query(self, q: str):
        return self.tp.query(q)
    
    def query_pandas(self, q: str) -> pd.DataFrame:
        return self.__query(q).as_pandas_dataframe()

    def query_single(self, q: str) -> Dict[str, Any]:
        return next(self.__query(q)).__dict__

    def execute(self, q: str):
        self.__query(q)

    def __create_thread_state_int_view(self):
        # thread state as intervals
        self.execute("""
            CREATE VIEW thread_state_int AS
            SELECT
                *,
                ts AS start,
                ts + dur AS end
            FROM thread_state
        """)

    def __create_freq_counter_view(self):
        self.execute("""
            CREATE VIEW freq_counter AS
            SELECT
                ts AS counter_start,
                /* use max signed int64 as the end ts for the last counter sample*/
                LEAD(ts, 1, 9223372036854775807) OVER (PARTITION BY cpu ORDER BY ts) AS counter_end,
                (LEAD(ts, 1, 9223372036854775807) OVER (PARTITION BY cpu ORDER BY ts) - ts) AS counter_dur,
                value AS freq_khz,
                cpu
            FROM counter
            INNER JOIN cpu_counter_track ON counter.track_id=cpu_counter_track.id
            WHERE cpu_counter_track.name='cpufreq'
        """)

    def __create_gc_slice_view(self):
        self.execute("""
            CREATE VIEW gc_slice AS
            SELECT
                slice.id AS slice_id,
                slice.ts AS slice_start,
                slice.dur AS slice_dur,
                slice.ts + slice.dur AS slice_end,
                slice.name AS slice_name,
                thread.utid,
                thread.name AS thread_name,
                process.upid,
                process.name AS process_name
            FROM slice
            INNER JOIN thread_track ON thread_track.id = slice.track_id
            INNER JOIN thread ON thread_track.utid = thread.utid
            INNER JOIN process ON thread.upid = process.upid
            WHERE slice.name LIKE '{}'
        """.format(LBO.GC_SLICE_SQL_LIKE))

    def __create_gc_slice_cpu_time_view(self):
        self.execute("""
            CREATE VIEW gc_slice_cpu_time AS
            SELECT
                gc_slice.upid,
                gc_slice.process_name,
                gc_slice.utid,
                gc_slice.thread_name,
                gc_slice.slice_id,
                gc_slice.slice_name,
                cpu,
                /* start and end of the overlap*/
                IIF(thread_state_int.start >= gc_slice.slice_start, thread_state_int.start, gc_slice.slice_start) AS sched_start,
                IIF(thread_state_int.end <= gc_slice.slice_end, thread_state_int.end, gc_slice.slice_end) AS sched_end
            FROM gc_slice
            INNER JOIN thread_state_int ON thread_state_int.utid = gc_slice.utid
            WHERE
                thread_state_int.state = 'Running' AND
                /* two intervals overlap */
                (
                    thread_state_int.start <= gc_slice.slice_end AND
                    gc_slice.slice_start <= thread_state_int.end
                )
        """)

    def __create_thread_cpu_time_view(self):
        self.execute("""
            CREATE VIEW thread_cpu_time AS
            SELECT
                process.upid,
                process.name AS process_name,
                thread.utid,
                thread.name AS thread_name,
                thread_state_int.cpu AS cpu,
                thread_state_int.start AS sched_start,
                thread_state_int.end AS sched_end
            FROM thread_state_int
            INNER JOIN thread ON thread.utid = thread_state_int.utid
            INNER JOIN process ON process.upid = thread.upid
            WHERE thread_state_int.state = 'Running'
        """)

    def __create_cycles_view(self, view_name: str, view: str):
        self.execute(f"""
            CREATE VIEW {view_name} AS
            SELECT
                {view}.*,
                /* intersect of two intervals */
                (IIF(freq_counter.counter_end <= {view}.sched_end, freq_counter.counter_end, {view}.sched_end) - /* min of the ends */
                IIF(freq_counter.counter_start >= {view}.sched_start, freq_counter.counter_start, {view}.sched_start)) * /* max of the starts */
                freq_counter.freq_khz / 1000000 AS cycles
            FROM {view}
            INNER JOIN freq_counter ON freq_counter.cpu = {view}.cpu
            WHERE
                /* two intervals overlap */
                (
                    freq_counter.counter_start <= {view}.sched_end AND
                    {view}.sched_start <= freq_counter.counter_end
                )
        """)

    def get_gc_slices(self, process: Optional[str] = None) -> pd.DataFrame:
        # Enable scheduling and atrace_categories: "dalvik"
        return self.query_pandas("""
            SELECT *
            FROM gc_slice
            {}
        """.format("WHERE process_name = '{}'".format(process) if process else ""))

    def get_gc_slices_cpu_time(self, process: Optional[str] = None) -> pd.DataFrame:
        # Enable scheduling and atrace_categories: "dalvik"
        return self.query_pandas("""
            SELECT
                upid,
                process_name,
                utid,
                thread_name,
                slice_id,
                slice_name,
                cpu,
                SUM(sched_end - sched_start) AS cpu_time
            FROM gc_slice_cpu_time
            {}
            GROUP BY gc_slice_cpu_time.slice_id, gc_slice_cpu_time.cpu
        """.format("WHERE process_name = '{}'".format(process) if process else ""))

    def get_process_cpu_time(self, process: str) -> pd.DataFrame:
        return self.query_pandas("""
            SELECT
                cpu,
                SUM(sched_end - sched_start) AS cpu_time
            FROM thread_cpu_time
            WHERE process_name = '{}'
            GROUP BY cpu
        """.format(process))

    def get_gc_slices_cycles(self, process: str) -> pd.DataFrame:
        return self.query_pandas(f"""
            SELECT
                upid,
                process_name,
                utid,
                thread_name,
                slice_id,
                slice_name,
                cpu,
                SUM(cycles) AS cycles
            FROM gc_slice_cycles
            WHERE process_name = '{process}'
            GROUP BY slice_id, cpu
        """)

    def get_process_cycles(self, process: str) -> pd.DataFrame:
        return self.query_pandas(f"""
            SELECT
                cpu,
                SUM(cycles) AS cycles
            FROM thread_cycles
            WHERE process_name = '{process}'
            GROUP BY cpu
        """)

    def distillation(self, process: str) -> pd.DataFrame:
        gc_df = self.get_gc_slices_cycles(process)[["cpu", "cycles"]].groupby("cpu").sum()
        process_df = self.get_process_cycles(process)[["cpu", "cycles"]].groupby("cpu").sum()
        df = pd.concat(gc_df.align(process_df, fill_value = 0), axis='columns')
        df.columns = ["distilled", "total"]
        return df


    def cpufreq(self) -> pd.DataFrame:
        return self.query_pandas("""
            SELECT * FROM freq_counter
        """)

    def integrate_mem(self, process: str, mem_type: str = "mem.rss.anon") -> int:
        # Enable ftrace_events: "kmem/rss_stat" in Perfetto
        # mem_type can be "mem.rss.anon", "mem.rss.file", "mem.rss.shmem", "mem.swap"
        df = self.tp.query("""
            SELECT ts, value, process_counter_track.name, process.name
            FROM counter
            INNER JOIN process_counter_track ON process_counter_track.id = counter.track_id
            INNER JOIN process ON process.upid = process_counter_track.upid
            WHERE process_counter_track.name = '{}' AND process.name = '{}'
            ORDER BY ts ASC
        """.format(mem_type, process)).as_pandas_dataframe()
        print(df)
        # value is in byte
        # ts is in nanosecond
        return scipy.integrate.trapezoid(df["value"], df["ts"]) / 1e9 / 1024 / 1024 # MB * second

In [ ]:
lbo = LBO("example-SS.perfetto-trace")

In [ ]:
lbo.get_gc_slices("com.google.android.apps.maps")

In [ ]:
lbo.get_gc_slices_cpu_time("com.google.android.apps.maps")

In [ ]:
lbo.distillation("com.google.android.apps.maps")